# tm_final_33 - Financial Tweet Sentiment Classification
## Text Mining 2025/2026 - NOVA IMS · group_33
### Final Solution — Restart & Run All

**Submitted model (single pipeline, single classification model — Guidelines 5.2):**
one `nickmuchi/finbert-tone-finetuned-fintwitter-classification` (FinBERT, 110M params)
trained by **knowledge distillation** (Hinton et al., 2015) from an 8-encoder weighted soft-vote
teacher (OOF F1-macro 92.01%; the ensemble itself is reported as extra work in `tm_tests_33.ipynb`).

- **Preprocessing:** `fix_tweet` — `ftfy` mojibake repair + truncation-artefact/URL cleanup (alters 50.1% of tweets).
- **Protocol:** 10-fold stratified CV, seed 42, best-checkpoint-per-fold, fp16 + GradScaler, cosine LR warmup.
- **Loss:** `(1-α)·weighted_CE + α·T²·KL(student‖teacher)`, α=0.5, T=2; teacher targets are the ensemble's leak-free OOF probabilities.

| Configuration | OOF F1-macro |
|---|---|
| Best plain single model (FinBERT 10ep fix_text) | 0.9082 |
| 8-model ensemble (teacher — extra work, not submitted) | 0.9201 |
| **Distilled single model (SUBMITTED)** | **0.9134** |

**Runtime:** ~2 min with the cached probabilities shipped in `results/predictions/`; ~40 min on a CUDA GPU to re-distil from the committed teacher OOF CSVs (load-or-train in Cell 6). **Instructions:** Kernel → Restart Kernel and Run All Cells. Produces `pred_33.csv`.

In [1]:
# Cell 1: Imports and reproducibility
import os, sys, re, time, gc, json, warnings, tempfile
warnings.filterwarnings('ignore')
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import ftfy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, accuracy_score, precision_score,
                             recall_score, classification_report)
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_cosine_schedule_with_warmup)

SEED = 42

def seed_all(seed=SEED):
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all()
print(f'Seed fixed: {SEED}')
print(f'Python: {sys.version[:20]} | torch {torch.__version__}')
print(f'ftfy {ftfy.__version__}')

Seed fixed: 42
Python: 3.11.9 (tags/v3.11.9 | torch 2.12.0+cu130
ftfy 6.3.1


In [2]:
# Cell 3: Preprocessing — ftfy mojibake fix + truncation cleanup
# Text-quality audit of the source CSV: 7.0% of tweets carry UTF-8 mojibake
# (e.g. â€™ instead of '), 16.4% end in truncation artefacts and 46.8% contain
# URLs with no sentiment value. fix_tweet repairs/removes these (alters 50.1%
# of tweets). Ablation at 10-fold: +0.26pp OOF F1-macro on FinBERT vs raw text.

_TRUNC_RE = re.compile(r'[�…°�…]+\s*(https?://\S*)?$')
_URL_RE    = re.compile(r'https?://\S+')
_TRAIL_RE  = re.compile(r'[\s\-–:]+$')

def fix_tweet(text: str) -> str:
    """Fix mojibake (ftfy) and remove truncation artefacts + bare URLs."""
    text = ftfy.fix_text(str(text))
    text = _TRUNC_RE.sub('', text)
    text = _URL_RE.sub('', text)
    return _TRAIL_RE.sub('', text).strip()

train = pd.read_csv('data/raw/train.csv')
test  = pd.read_csv('data/raw/test.csv')

texts      = [fix_tweet(t) for t in train['text'].tolist()]
test_texts = [fix_tweet(t) for t in test['text'].tolist()]
y = train['label'].values

print(f'Train: {train.shape} | Test: {test.shape}')
print('Label distribution (0=Bearish, 1=Bullish, 2=Neutral):')
print(train['label'].value_counts().sort_index())
print(f'\nSample before: {train["text"].iloc[1][:90]}')
print(f'Sample after:  {texts[1][:90]}')

Train: (9543, 2) | Test: (2388, 2)
Label distribution (0=Bearish, 1=Bullish, 2=Neutral):
label
0    1442
1    1923
2    6178
Name: count, dtype: int64

Sample before: $CCL $RCL - Nomura points to bookings weakness at Carnival and Royal Caribbean https://t.c
Sample after:  $CCL $RCL - Nomura points to bookings weakness at Carnival and Royal Caribbean


In [3]:
# Cell 4: Configuration and device selection
TAG          = 'finbert_distilled'   # the submitted single model (distilled student)
MODEL_NAME   = 'nickmuchi/finbert-tone-finetuned-fintwitter-classification'
MAXLEN       = 128
LR           = 5e-6
WEIGHT_DECAY = 0.01
EPOCHS       = 10
N_FOLDS      = 10
WARMUP_RATIO = 0.06
KD_ALPHA     = 0.5    # weight of the distillation (KL) term in the loss
KD_TEMP      = 2.0    # distillation temperature
GRAD_ACCUM   = 1

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    BATCH, EVAL_BATCH = 16, 32
    AMP_DTYPE = torch.float16
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    USE_SCALER = True
    print(f'GPU: {torch.cuda.get_device_name(0)} | '
          f'VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB | '
          f'fp16 + GradScaler')
else:
    BATCH, EVAL_BATCH = 8, 32
    AMP_DTYPE, USE_SCALER = None, False
    torch.set_num_threads(min(12, os.cpu_count()))
    print('CPU fallback')

# Inverse-frequency class weights
counts = np.bincount(y, minlength=3)
class_weights = torch.tensor(len(y) / (3 * counts), dtype=torch.float32)
print(f'class weights: {[round(x,3) for x in class_weights.tolist()]}')

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

GPU: NVIDIA GeForce RTX 5070 | VRAM 12.8 GB | fp16 + GradScaler
class weights: [2.206, 1.654, 0.515]


In [4]:
# Cell 5: Model builder, distillation-aware training loop and inference helpers
# Loss: (1-a)*weighted_CE(hard labels) + a*T^2*KL(student_T || teacher_T)
# If no teacher targets are provided, falls back to plain weighted CE.
import torch.nn.functional as F

def build_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3, ignore_mismatched_sizes=True,
        dtype=torch.float32,   # fp32 params required for GradScaler
    ).to(DEVICE)


@torch.no_grad()
def predict_proba(model, txts):
    model.eval()
    out = []
    for i in range(0, len(txts), EVAL_BATCH):
        enc = tok(txts[i:i+EVAL_BATCH], padding=True, truncation=True,
                  max_length=MAXLEN, return_tensors='pt')
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        if DEVICE == 'cuda' and AMP_DTYPE is not None:
            with torch.autocast(device_type='cuda', dtype=AMP_DTYPE):
                logits = model(**enc).logits
        else:
            logits = model(**enc).logits
        out.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
    return np.vstack(out)


def run_fold(tr_texts, tr_labels, va_texts, va_y, all_test_texts, fold, tr_soft=None):
    """Train one fold (optionally distilling from teacher soft targets `tr_soft`).
    Returns (best_va_proba, best_test_proba, best_f1)."""
    seed_all(SEED + fold)
    model = build_model()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    steps_per_epoch = int(np.ceil(len(tr_texts) / BATCH))
    total_steps = steps_per_epoch * EPOCHS
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, int(WARMUP_RATIO * total_steps), total_steps)
    ce_loss = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
    scaler = torch.amp.GradScaler('cuda') if USE_SCALER else None
    T, a = KD_TEMP, KD_ALPHA

    best_f1, best_va_proba = -1.0, None
    n = len(tr_texts)

    with tempfile.TemporaryDirectory() as tmpdir:
        ckpt = Path(tmpdir) / 'best.pt'
        for epoch in range(EPOCHS):
            model.train()
            order = np.random.permutation(n)
            running, t0 = 0.0, time.time()
            for i in range(0, n, BATCH):
                bidx = order[i:i+BATCH]
                bt = [tr_texts[j] for j in bidx]
                bl = torch.tensor([tr_labels[j] for j in bidx], dtype=torch.long, device=DEVICE)
                enc = tok(bt, padding=True, truncation=True, max_length=MAXLEN, return_tensors='pt')
                enc = {k: v.to(DEVICE) for k, v in enc.items()}
                optimizer.zero_grad(set_to_none=True)

                def compute_loss():
                    logits = model(**enc).logits
                    loss = (1 - a) * ce_loss(logits, bl) if tr_soft is not None else ce_loss(logits, bl)
                    if tr_soft is not None:
                        bsoft = tr_soft[bidx].to(DEVICE)
                        log_s = F.log_softmax(logits / T, dim=1)
                        loss = loss + a * (T * T) * F.kl_div(log_s, bsoft, reduction='batchmean')
                    return loss

                if DEVICE == 'cuda' and AMP_DTYPE is not None:
                    with torch.autocast(device_type='cuda', dtype=AMP_DTYPE):
                        loss = compute_loss()
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer); scaler.update()
                else:
                    loss = compute_loss()
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                scheduler.step()
                running += float(loss.item())

            va_proba = predict_proba(model, va_texts)
            ep_f1 = f1_score(va_y, va_proba.argmax(1), average='macro')
            star = ' *** best ***' if ep_f1 > best_f1 else ''
            print(f'    fold {fold+1} epoch {epoch+1}/{EPOCHS} '
                  f'loss={running/steps_per_epoch:.4f} val_f1={ep_f1:.6f} '
                  f'({time.time()-t0:.0f}s){star}')
            if ep_f1 > best_f1:
                best_f1 = ep_f1
                best_va_proba = va_proba.copy()
                torch.save(model.state_dict(), ckpt)
        # reload best checkpoint for test predictions
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        best_test_proba = predict_proba(model, all_test_texts)

    print(f'  fold {fold+1} best val macro-F1={best_f1:.6f}')
    return best_va_proba, best_test_proba, best_f1

print('Helpers defined (distillation-aware).')

Helpers defined (distillation-aware).


In [5]:
# Cell 6: LOAD-OR-TRAIN — the submitted single model (distilled FinBERT)
# If cached OOF/test probabilities exist (shipped in results/predictions/), load them (~2 s).
# Otherwise re-distil from the committed teacher OOF CSVs (~40 min on GPU).
# If the teacher files are also absent, trains the same recipe with plain weighted CE.

OOF_PATH     = Path(f'results/predictions/oof_proba_{TAG}.npy')
OOF_CSV      = Path(f'results/predictions/oof_proba_{TAG}.csv')
TEST_PATH    = Path(f'results/predictions/test_proba_{TAG}.npy')
CSV_PATH     = Path(f'results/predictions/prob_test_{TAG}.csv')
JSON_PATH    = Path(f'results/tables/{TAG}_result.json')
WEIGHTS_PATH = Path('results/tables/ensemble_optimal_result.json')

if (OOF_PATH.exists() or OOF_CSV.exists()) and (TEST_PATH.exists() or CSV_PATH.exists()):
    # ---- cached path -------------------------------------------------------
    oof_proba = (np.load(OOF_PATH) if OOF_PATH.exists()
                 else pd.read_csv(OOF_CSV)[['p0','p1','p2']].values).astype(np.float32)
    test_proba_main = (np.load(TEST_PATH) if TEST_PATH.exists()
                       else pd.read_csv(CSV_PATH)[['p0','p1','p2']].values).astype(np.float32)
    result_cached = json.loads(JSON_PATH.read_text()) if JSON_PATH.exists() else {}
    fold_f1 = result_cached.get('per_fold_f1', [])
    print(f'[CACHED] {TAG}')
    print(f'  OOF proba  : {oof_proba.shape}  | test proba: {test_proba_main.shape}')
    if fold_f1:
        print(f'  per-fold F1: {[round(x,4) for x in fold_f1]}')
else:
    # ---- training path: build teacher soft targets, then distil -------------
    teacher_soft = None
    if WEIGHTS_PATH.exists():
        opt = json.loads(WEIGHTS_PATH.read_text())
        t_sum, t_w = np.zeros((len(y), 3)), 0.0
        for tag_t, w in opt['models'].items():
            for p in [Path(f'results/predictions/oof_proba_{tag_t}.npy'),
                      Path(f'results/predictions/oof_proba_{tag_t}.csv')]:
                if p.exists():
                    arr = (np.load(p) if p.suffix == '.npy'
                           else pd.read_csv(p)[['p0','p1','p2']].values)
                    t_sum += arr * w; t_w += w
                    break
        if t_w > 0:
            teacher = t_sum / t_w
            print(f'Teacher: {t_w:.2f} total weight, OOF F1-macro='
                  f'{f1_score(y, teacher.argmax(1), average="macro"):.4f}')
            teacher_soft = torch.softmax(
                torch.log(torch.tensor(teacher, dtype=torch.float32) + 1e-9) / KD_TEMP, dim=1)
    if teacher_soft is None:
        print('Teacher OOF files not found — training with plain weighted CE instead.')

    print(f'[TRAINING] {TAG} — {N_FOLDS}-fold from scratch...')
    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof_proba      = np.zeros((len(y), 3), dtype=np.float32)
    test_proba_sum = np.zeros((len(test_texts), 3), dtype=np.float32)
    fold_f1 = []
    t_start = time.time()

    for fold, (tr_idx, va_idx) in enumerate(cv.split(texts, y)):
        print(f'\n=== Fold {fold+1}/{N_FOLDS} ===')
        tr_soft = teacher_soft[tr_idx] if teacher_soft is not None else None
        va_proba, test_proba_fold, best_f1 = run_fold(
            [texts[i] for i in tr_idx], [int(y[i]) for i in tr_idx],
            [texts[i] for i in va_idx], y[va_idx], test_texts, fold, tr_soft=tr_soft)
        oof_proba[va_idx] = va_proba
        test_proba_sum   += test_proba_fold
        fold_f1.append(best_f1)
        gc.collect()
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

    test_proba_main = test_proba_sum / N_FOLDS
    # save cache so the next Restart & Run All loads instantly
    OOF_PATH.parent.mkdir(parents=True, exist_ok=True)
    np.save(OOF_PATH, oof_proba)
    np.save(TEST_PATH, test_proba_main)
    pd.DataFrame(test_proba_main, columns=['p0','p1','p2']).to_csv(CSV_PATH, index=False)
    print(f'\nDone in {(time.time()-t_start)/60:.1f} min. Cache saved.')

[CACHED] finbert_distilled
  OOF proba  : (9543, 3)  | test proba: (2388, 3)
  per-fold F1: [0.9101, 0.9078, 0.8999, 0.9093, 0.9259, 0.9369, 0.9155, 0.9104, 0.9116, 0.9072]


In [6]:
# Cell 7: Out-of-fold performance of the primary model (honest, leak-free estimate)
oof_pred = oof_proba.argmax(1)
oof_f1   = f1_score(y, oof_pred, average='macro')

fold_info = (f'(per-fold {[round(x,4) for x in fold_f1]}, std {np.std(fold_f1):.4f})'
             if fold_f1 else '(per-fold scores in results/tables JSON)')
print(f'OOF F1-macro : {oof_f1:.4f}  {fold_info}')
print(f'Accuracy     : {accuracy_score(y, oof_pred):.4f}')
print(f'Precision-mac: {precision_score(y, oof_pred, average="macro"):.4f}')
print(f'Recall-macro : {recall_score(y, oof_pred, average="macro"):.4f}')
print()
print(classification_report(y, oof_pred, target_names=['Bearish', 'Bullish', 'Neutral']))

OOF F1-macro : 0.9134  (per-fold [0.9101, 0.9078, 0.8999, 0.9093, 0.9259, 0.9369, 0.9155, 0.9104, 0.9116, 0.9072], std 0.0100)
Accuracy     : 0.9326
Precision-mac: 0.9017
Recall-macro : 0.9263

              precision    recall  f1-score   support

     Bearish       0.84      0.91      0.87      1442
     Bullish       0.90      0.93      0.92      1923
     Neutral       0.97      0.94      0.95      6178

    accuracy                           0.93      9543
   macro avg       0.90      0.93      0.91      9543
weighted avg       0.93      0.93      0.93      9543



In [7]:
# Cell 7b: Submission compliance — the final prediction uses ONLY the single model above
# Guidelines 5.2: "a single pipeline with a single classification model".
# The 8-encoder ensemble (OOF F1 92.01%) is EXTRA WORK reported in tm_tests_33.ipynb;
# its knowledge reaches this submission only through the distillation loss of Cell 5/6.

test_proba_final = test_proba_main   # single distilled model — no ensembling here

print('Submitted solution: ONE model —', MODEL_NAME)
print(f'  trained with knowledge distillation (alpha={KD_ALPHA}, T={KD_TEMP})')
print(f'  Single-model OOF F1-macro: {oof_f1:.4f}')
if Path('results/tables/ensemble_optimal_result.json').exists():
    _opt = json.loads(Path('results/tables/ensemble_optimal_result.json').read_text())
    print(f'  (teacher ensemble OOF F1-macro: {_opt["oof_f1_macro"]:.4f} — extra work, not submitted)')

Submitted solution: ONE model — nickmuchi/finbert-tone-finetuned-fintwitter-classification
  trained with knowledge distillation (alpha=0.5, T=2.0)
  Single-model OOF F1-macro: 0.9134
  (teacher ensemble OOF F1-macro: 0.9201 — extra work, not submitted)


In [8]:
# Cell 8: Generate predictions and save pred_33.csv (with full validation)
test_pred  = test_proba_final.argmax(1)
submission = pd.DataFrame({'id': test['id'], 'label': test_pred.astype(int)})
submission.to_csv('pred_33.csv', index=False)
os.makedirs('results/predictions', exist_ok=True)
submission.to_csv('results/predictions/pred_final_distilled.csv', index=False)

print(f'Predictions saved: pred_33.csv ({len(submission)} rows)')
print('Distribution:')
print(submission['label'].value_counts().sort_index()
      .rename({0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}))
print()

# Robust validation of the deliverable
assert len(submission) == 2388,                        f'Expected 2388, got {len(submission)}'
assert list(submission.columns) == ['id', 'label'],    'Columns must be exactly [id, label]'
assert set(submission['label'].unique()) == {0, 1, 2}, f'Labels must be {{0,1,2}}'
assert submission['label'].isna().sum() == 0,          'NaN labels found!'
assert submission['id'].is_unique,                     'Duplicate ids found!'
assert (submission['label'] >= 0).all() and (submission['label'] <= 2).all(), 'Labels out of range'
min_class_pct = submission['label'].value_counts(normalize=True).min()
assert min_class_pct > 0.01, f'Under-represented class ({min_class_pct:.1%}) — possible model collapse'

print('All assertions PASSED.')
print()
print(submission.head(10).to_string(index=False))

Predictions saved: pred_33.csv (2388 rows)
Distribution:
label
Bearish     393
Bullish     495
Neutral    1500
Name: count, dtype: int64

All assertions PASSED.

 id  label
  0      1
  1      2
  2      2
  3      2
  4      2
  5      1
  6      2
  7      0
  8      2
  9      2
